[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/svngoku/cybersup-deep-learning-intro/blob/main/notebooks/formateur/02_cnn_vision_corrige.ipynb)


# TP 02 — CNN sur digits
**150 minutes · CPU · dix classes**

Calculer les formes et les paramètres avant d'exécuter, puis effectuer une ablation.
Le jeu intégré à scikit-learn évite tout téléchargement. Les niveaux de gris vont de 0 à 16.
Découpage : protocole 20 min, dimensions 30 min, entraînement 40 min, ablation 35 min, analyse 25 min.

**VERSION FORMATEUR — réponses et points de contrôle en fin de notebook.**

## Préparation de l'environnement

Exécuter cette cellule en premier dans un runtime Python 3 sur **CPU**. Les bibliothèques déjà installées sont conservées ; seules celles qui manquent sont ajoutées. Aucun clonage, fichier voisin ou montage Drive n'est nécessaire. Enregistrer une copie du notebook avant de commencer. Si le lien vers le dépôt privé est inaccessible, importer le fichier fourni dans le pack étudiant.


In [ ]:
# Cellule autonome : les bibliothèques déjà présentes dans Colab sont conservées.
import importlib.util
import json
import os
import platform
import subprocess
import sys

DEPENDENCIES = {
    "numpy": "numpy==2.2.6",
    "sklearn": "scikit-learn==1.7.2",
    "torch": "torch==2.8.0",
    "matplotlib": "matplotlib==3.10.6",
}
missing = [pin for module, pin in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des bibliothèques absentes :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import matplotlib
import numpy as np
import sklearn
import torch

ENVIRONMENT = {
    "runtime": "google-colab" if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ else "python-local-ou-ci",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
    "matplotlib": matplotlib.__version__,
    "device": "cpu",
    "cuda_available": torch.cuda.is_available(),
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))
print("Les cinq TP utilisent le CPU, même si un GPU est disponible.")


In [ ]:
import os, copy, math, random
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device('cpu')
print('Référence : CPU | PyTorch', torch.__version__, '| graine', SEED)

In [ ]:
def evaluate(model, X, y):
    model.eval()
    with torch.inference_mode():
        logits = model(X)
        loss = nn.functional.cross_entropy(logits, y).item()
        pred = logits.argmax(-1).numpy()
    return loss, accuracy_score(y.numpy(), pred), pred

def fit(model, X, y, Xv, yv, epochs=12, lr=1e-3, frozen=False):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    best_loss = float('inf'); best = copy.deepcopy(model.state_dict()); history=[]
    gen = torch.Generator().manual_seed(SEED)
    for epoch in range(epochs):
        model.train()
        if frozen: model.features.eval()
        total=0.0
        for ids in torch.randperm(len(y), generator=gen).split(64):
            opt.zero_grad(set_to_none=True)
            loss=nn.functional.cross_entropy(model(X[ids]),y[ids])
            loss.backward(); opt.step(); total += loss.item()*len(ids)
        vl,va,_=evaluate(model,Xv,yv)
        history.append((total/len(y),vl,va))
        if vl < best_loss: best_loss=vl; best=copy.deepcopy(model.state_dict())
    model.load_state_dict(best)
    return np.array(history)

class TinyCNN(nn.Module):
    def __init__(self, classes=10):
        super().__init__()
        self.features=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                                   nn.Conv2d(8,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten())
        self.head=nn.Linear(64,classes)
    def forward(self,x): return self.head(self.features(x))

In [ ]:
digits=load_digits(); X=digits.images.astype('float32')[:,None]/16.0; y=digits.target
ids=np.arange(len(y))
it,ite=train_test_split(ids,test_size=.15,stratify=y,random_state=SEED)
itr,iv=train_test_split(it,test_size=.15/.85,stratify=y[it],random_state=SEED)
assert not(set(itr)&set(iv) or set(itr)&set(ite) or set(iv)&set(ite))
Xtr,Xval,Xtest=[torch.tensor(X[i]) for i in (itr,iv,ite)]
ytr,yval,ytest=[torch.tensor(y[i],dtype=torch.long) for i in (itr,iv,ite)]
model=TinyCNN()
print(model)
print('Paramètres :',sum(p.numel() for p in model.parameters()))
assert sum(p.numel() for p in model.parameters())==1898
assert model.features(Xtr[:4]).shape==(4,64)
assert model(Xtr[:4]).shape==(4,10)

In [ ]:
history=fit(model,Xtr,ytr,Xval,yval,epochs=15)
plt.plot(history[:,0],label='train');plt.plot(history[:,1],label='validation')
plt.xlabel('Époque');plt.ylabel('Entropie croisée');plt.legend();plt.show()
vl,va,predv=evaluate(model,Xval,yval)
print('Validation :',vl,va)
ConfusionMatrixDisplay.from_predictions(yval.numpy(),predv);plt.title('Validation');plt.show()

## Investigations
1. Refaire le tableau 1×8×8 → 8×8×8 → 8×4×4 → 16×4×4 → 16×2×2 → 64 → 10.
2. Remplacer le max pooling par une réduction utilisant le stride ; corriger les formes et recompter les poids.
3. Comparer une configuration à la base avec le même split, les mêmes époques et la même règle de sélection.
4. Montrer trois erreurs de validation et proposer une hypothèse par erreur.
5. Mesurer le temps sur votre machine et préciser le nombre de threads.

La séparation est stratifiée par classe ; elle ne garantit pas une séparation par auteur des chiffres.

In [ ]:
loss,acc,pred=evaluate(model,Xtest,ytest)
RESULT={'tp':2,'parameters':sum(p.numel() for p in model.parameters()),'test_loss':loss,
        'test_accuracy':acc,'macro_f1':f1_score(ytest.numpy(),pred,average='macro')}
print(RESULT)

## Corrigé — dimensions et protocole
Les deux convolutions contiennent respectivement 80 et 1168 paramètres ; la tête contient 650 paramètres, total 1898.
Un max pooling n'ajoute pas de poids. La sortie de chaque pooling divise les dimensions spatiales par deux.
Une convolution 1→8, k=3, p=1, s=2 donne directement 8×4×4 ; une autre 8→16, s=2 donne 16×2×2.
Cette modification conserve les paramètres des convolutions mais change l'opération de sous-échantillonnage.
La matrice de confusion affiche les vraies classes en lignes. Une comparaison à une graine reste indicative.
Le score test est terminal : les décisions d'architecture s'appuient sur la validation.

## Exporter les résultats

Exécuter la cellule suivante après le bilan terminal pour enregistrer métriques, graine et versions dans un fichier JSON. Dans Colab, activer `DOWNLOAD_RESULTS` pour le télécharger. Sauvegarder également une copie du notebook avec les sorties et les réponses aux investigations. Les fichiers du runtime sont temporaires.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

DOWNLOAD_RESULTS = False  # @param {type:"boolean"}

def json_scalar(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Valeur non sérialisable : {type(value).__name__}")

result_path = Path("resultats") / f"tp_{RESULT['tp']:02d}.json"
result_path.parent.mkdir(exist_ok=True)
experiment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "environment": ENVIRONMENT,
    "results": RESULT,
}
result_path.write_text(
    json.dumps(experiment, indent=2, ensure_ascii=False, default=json_scalar) + "\n",
    encoding="utf-8",
)
print("Résultats enregistrés :", result_path)

if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print("Hors Colab, récupérer le fichier dans le dossier resultats/.")
    else:
        files.download(str(result_path))
